In [1]:
import os
import json
import math
import random
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

from stgcn.st_gcn import Model

In [2]:
DSET_PATH = "/Users/steventan/.cache/kagglehub/datasets/soumicksarker/ipn-hand-dataset/versions/7"
TRAIN_PATH = os.path.join(DSET_PATH, "train_skeletons")
TEST_PATH = os.path.join(DSET_PATH, "test_skeletons")

def load_skel_data(path):
    skeletons = torch.load(os.path.join(path, "skeletons_tensor.pt"))  
    with open(os.path.join(path, "skeleton_annots.json"), "r") as f:
        metadata = json.load(f)
    return skeletons, metadata

def reshape_skel_for_stgcn(skels):
    return skels.permute(0, 3, 1, 2).unsqueeze(-1)

train_skels, train_metadata = load_skel_data(TRAIN_PATH)
test_skels, test_metadata = load_skel_data(TEST_PATH)

# reshape skeleton data for the format used by st-gcn paper
train_skels = reshape_skel_for_stgcn(train_skels)
test_skels = reshape_skel_for_stgcn(test_skels)

# minus one to make 0-indexed
train_labels = torch.tensor(
    [sample["label_id"] - 1 for sample in train_metadata["samples"]],
    dtype=torch.long,
)
test_labels = torch.tensor(
    [sample["label_id"] - 1 for sample in test_metadata["samples"]],
    dtype=torch.long,
)


# filter out first 100 training datapoints since they aren't labelled well
train_skels = train_skels[100:]
train_labels = train_labels[100:]

num_classes = len(torch.unique(train_labels))

# sanity check stuff
print("Labels shape:", train_labels.shape)
print("Skeletons shape:", train_skels.shape)
print("Num classes:", num_classes)

Labels shape: torch.Size([3781])
Skeletons shape: torch.Size([3781, 3, 210, 21, 1])
Num classes: 14


/var/folders/_1/ph20kymj4s9bnnn49x3mjlxr0000gn/T/ipykernel_1244/1268182783.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  skeletons = torch.load(os.path.join(path, "ske

In [3]:
# Set device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [4]:
train_ds = TensorDataset(train_skels, train_labels)
test_ds = TensorDataset(test_skels, test_labels)

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print("Train samples:", len(train_ds))
print("Test samples:", len(test_ds))

Train samples: 3781
Test samples: 1556


In [14]:
def run_epoch(loader, model, criterion, optimizer=None):
    if optimizer is None:
        model.eval()
    else:
        model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in loader:    
        if optimizer is not None:
            # Training mode
            optimizer.zero_grad()

        x_batch = x_batch.to(device, dtype=torch.float32)
        y_batch = y_batch.to(device)
    
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
    
        if optimizer is not None:
            loss.backward()
            optimizer.step()
    
        total_loss += loss.item() * x_batch.size(0)
    
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    
    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


def train_model(train_loader, test_loader, model, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    train_accs = []
    test_accs = []
    
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer)
        test_loss, test_acc = run_epoch(test_loader, model, criterion, optimizer=None)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.3f}"
        )

    return train_accs, test_accs

def eval_model(model, test_loader):
    criterion = nn.CrossEntropyLoss()
    loss, acc = run_epoch(test_loader, model, criterion)
    print(f"Test Accuracy: {acc}")

In [15]:
model = Model(num_class=num_classes, graph_args={}, in_channels=3, edge_importance_weighting=False, learn_graph=True, dropout=0.5)
model = model.to(device)
train_accs, test_accs = train_model(train_loader, test_loader, model, epochs=50)

Epoch 01 | train_loss=1.8977, train_acc=0.401 | test_loss=1.8079, test_acc=0.437
Epoch 02 | train_loss=1.2366, train_acc=0.611 | test_loss=1.2082, test_acc=0.674


KeyboardInterrupt: 

In [ ]:
model_name = "stgcn_graph_dropout_0.5"
torch.save(model.state_dict(), f"{model_name}.pth")

In [17]:
model = Model(num_class=num_classes, graph_args={}, in_channels=3, learn_graph=True, edge_importance_weighting=False)
model.load_state_dict(torch.load("stgcn_graph_dropout_0.5.pth"))
model.to(device)
None

/var/folders/_1/ph20kymj4s9bnnn49x3mjlxr0000gn/T/ipykernel_1244/2158324917.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("stgcn_graph_

In [18]:
eval_model(model, test_loader)

Test Accuracy: 0.8708226221079691


In [ ]:
x = np.arange(50)
plt.figure(figsize=(8, 5))
plt.plot(x, train_accs, label='Train Accuracy')
plt.plot(x, test_accs, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy for STGCN Model')
plt.legend()
plt.grid(True)
plt.savefig(f"{model_name}.png")